# 11장. Guardrails와 사람의 승인 구현하기

기본 CSV 보고서의 로컬 등록을 정책과 사용자 판단으로 제어합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | 입력·보고서 검사·채널 정책·사람 승인 |
| 이번 실습 조합 | 기본 보고서 + publish_report 함수 + 로컬 등록 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

| 판단 | 실행 주체 | 확인할 내용 |
| --- | --- | --- |
| 입력 검사 | 노트북 애플리케이션 | CSV 컬럼·빈 데이터·정수·음수 |
| 보고서 검사 | 노트북 애플리케이션 | 제목 형식·검토 대상 바이트와 SHA-256 |
| 등록 정책 | PublicationGate | 역할·보고서 ID·채널 |
| 업무 승인 | 사용자 callback | 이 보고서를 로컬에 등록할지 |
| Automatic review | Agents API | 지원되는 실행 권한 요청 |

기본 APPROVAL_MODE는 interactive이며 APPROVE와 REJECT 모두 정상 결과입니다. REVIEW_MODE는 별도 설정이며 기본 none으로 업무 승인 실습을 진행합니다. auto_review는 프로젝트에서 사용 가능한 경우에만 선택합니다. review를 none으로 두어도 사용자 callback은 실행됩니다. 자동 검증은 `--approval-fixture`를 명시하고 사람 입력과 구분합니다.

수동 실행은 위에서 아래로 진행합니다. 자동 검증: `uv run --group dev python scripts/run_extensions.py --chapter 11 --live --approval-fixture`. 실제 외부 게시·이메일 전송은 없으며 승인 대상은 로컬 실습 폴더입니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 환경 준비와 실행 모드

로컬에서는 `uv sync --locked` 후 `.venv` 커널을 선택합니다. Colab에서는 다운로드 안내의 독립 실행본을 업로드합니다. 준비 셀은 패키지를 직접 설치하고 포함 파일을 생성합니다.
아래 RUN_LIVE를 True로 바꿨을 때만 실제 API 호출·sandbox 생성이 발생합니다.
기존 `.env.local`의 키·모델을 재사용하며 키는 sandbox·ZIP·로그에 넣지 않습니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import json
from tutorial_helpers.common import load_settings

RUN_LIVE = False
APPROVAL_MODE = "interactive"
REVIEW_MODE = "none"
MODEL_OVERRIDE = ""
CHAPTER = "11_guardrails_approvals"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print({"live_enabled": settings.enabled, "model": settings.model or "Offline mode",
       "approval_mode": APPROVAL_MODE, "review_mode": REVIEW_MODE})


## 1. 실제 입력과 정책 경계 확인하기

잘못된 예제의 거절과 실제 전송할 CSV의 통과를 따로 확인합니다. 다음 셀은 실제 바이트를 읽습니다. 입력 검사 함수는 정답 집계를 하지 않으며 계산값·보고서 내용 대조는 자동 검증 도구의 사본에서만 수행합니다.

역할 analyst는 신뢰하는 애플리케이션 문맥의 실습 값이며 사용자 인증 구현이 아닙니다. 모델의 approved 같은 필드를 받지 않습니다. report_id와 channel만 요청하게 하고 역할·승인 여부는 애플리케이션이 결정합니다.


In [ ]:
input_path = settings.root / "data/sales.csv"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
print(csv_text)


In [ ]:
from tutorial_helpers.common import new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error
from tutorial_helpers.extensions import ResultCache
from uuid import uuid4
import base64
import time
from pathlib import Path


## 2. 요청 도구와 승인 callback 정의하기

publish_report는 등록을 요청하는 Function Tool입니다. API가 이 Python 함수를 sandbox에서 자동 실행하는 것은 아닙니다. 노트북이 required action을 받아 로컬 정책을 실행하고 결과를 API로 반환합니다. [Functions](https://developers.openai.com/api/docs/guides/agents-api/tools/functions)

승인 callback의 핵심을 아래에 직접 정의합니다. UI 안내는 출력하고 실제 입력은 input으로 받습니다. 정확한 APPROVE만 승인하며 REJECT를 포함한 다른 입력은 거절합니다. KeyboardInterrupt는 승인 거절 대신 실행 중단입니다.


In [ ]:
from tutorial_helpers.extensions import PolicyError, Approval, validation_approval
import hashlib

PUBLISH_TOOL = {
    "type": "function",
    "name": "publish_report",
    "description": (
        "Request registration of a validated report. "
        "The application enforces channel, role and "
        "approval policy."
    ),
    "parameters": {
        "type": "object",
        "properties": {"report_id": {"type": "string"}, "channel": {"type": "string"}},
        "required": ["report_id", "channel"],
        "additionalProperties": False,
    },
}


In [ ]:
def interactive_approval(report_id, digest, channel):
    print("Action required: review the displayed report before deciding.", flush=True)
    print("VS Code: use the input box at the top of the window.", flush=True)
    print("Jupyter/Colab: use the notebook input prompt.", flush=True)
    print("Type APPROVE or REJECT, then press Enter. Input has no automatic timeout.", flush=True)
    entered = input(f"Register {report_id} in {channel}? SHA256={digest}. Type APPROVE: ")
    return Approval(entered == "APPROVE", report_id, digest, channel, "local-learner", "interactive")


### 정책은 사람에게 묻기 전에 실행합니다

| 순서 | 검사 | 실패 시 |
| --- | --- | --- |
| 1 | 도구 인자·역할·보고서 ID | 정책 오류 반환 |
| 2 | channel이 training인지 | 승인 질문 없이 channel_not_allowed |
| 3 | 파일이 검토한 바이트와 같은지 | report_changed |
| 4 | 사용자 판단·승인 대상 binding | 거절 또는 불일치 반환 |
| 5 | 승인 대기 중 파일 변경 여부 | 쓰기 전에 report_changed |
| 6 | 같은 바이트의 로컬 등록 | 승인 영수증·등록 결과 반환 |

관문은 처음 읽은 보고서 바이트를 고정하고 승인 전후 원본이 같은지 확인합니다. 변경되면 새 파일을 검토하고 새 실행을 시작합니다. 해시는 바이트 식별 정보이며 내용의 정확성·서명·DLP를 보장하지 않습니다. 로컬 파일과 영수증 저장은 분산 트랜잭션 구현이 아니므로 쓰기 실패 시 파일 상태를 조사해야 합니다.


In [ ]:
class PublicationGate:
    ("Local training storage only; no email, public" " URL, or external publication.")

    def __init__(
        self, report_id, path, destination, approver=interactive_approval, actor_role="analyst"
    ):
        raw = Path(path).read_bytes()
        if not raw.decode("utf-8").startswith("# 매출 분석 보고서"):
            raise PolicyError("invalid_report")
        self.report_id = report_id
        self.path = Path(path)
        self.reviewed_bytes = raw
        self.digest = hashlib.sha256(raw).hexdigest()
        self.destination = Path(destination)
        self.approver = approver
        self.actor_role = actor_role
        self.receipts = []
        self.decision = None

    def publish(self, arguments):
        if set(arguments) != {"report_id", "channel"}:
            raise PolicyError("invalid_arguments")
        if self.actor_role != "analyst":
            raise PolicyError("role_not_allowed")
        if arguments["report_id"] != self.report_id:
            raise PolicyError("unknown_report")
        if arguments["channel"] != "training":
            raise PolicyError("channel_not_allowed")
        raw = self.path.read_bytes()
        if raw != self.reviewed_bytes:
            raise PolicyError("report_changed")
        decision = self.approver(self.report_id, self.digest, "training")
        self.decision = decision
        if not decision.approved:
            raise PolicyError("approval_denied")
        if (decision.report_id, decision.sha256, decision.channel) != (
            self.report_id,
            self.digest,
            "training",
        ):
            raise PolicyError("approval_binding_mismatch")
        if self.path.read_bytes() != raw:
            raise PolicyError("report_changed")
        self.destination.mkdir(parents=True, exist_ok=True)
        receipt = {
            "report_id": self.report_id,
            "sha256": self.digest,
            "channel": "training",
            "actor": decision.actor,
            "approval_origin": decision.origin,
        }
        (self.destination / f"{self.digest}.approval.json").write_text(
            json.dumps(receipt, indent=2)
        )
        target = self.destination / f"{self.digest}.md"
        if target.exists():
            if target.read_bytes() != raw:
                raise PolicyError("stored_report_conflict")
        else:
            with target.open("xb") as output:
                output.write(raw)
        self.receipts.append(receipt)
        return {
            "publication_status": "published",
            "report_id": self.report_id,
            "sha256": self.digest,
            "channel": "training",
            "approval_origin": decision.origin,
        }


### 입력 검사와 두 실행 모드 선택

음수 예제는 데이터 치환 없이 별도로 만들고 거절을 확인합니다. 같은 함수로 실제 CSV도 검사해 통과해야 API 단계로 진행합니다. review의 auto_review는 프로젝트 지원이 필요하며 선택 불가 오류를 업무 승인 실패로 오해하지 마세요. 자동 fallback은 하지 않습니다. [Automatic review](https://developers.openai.com/api/docs/guides/agents-api/sessions/approvals)

fixture는 결정적인 테스트 승인이고 사람이 승인했다는 뜻이 아닙니다. CLI의 --approval-fixture는 실행 사본의 APPROVAL_MODE만 바꾸며 REVIEW_MODE는 별도입니다.


In [ ]:
import csv
import io


def validate_input(text):
    reader = csv.DictReader(io.StringIO(text))
    fields = ["date", "product", "quantity", "unit_price"]
    if reader.fieldnames != fields:
        raise PolicyError("invalid_csv")
    rows = list(reader)
    if not rows:
        raise PolicyError("invalid_csv")
    try:
        for row in rows:
            if set(row) != set(fields) or any(not row[field] for field in fields):
                raise PolicyError("invalid_csv")
            if int(row["quantity"]) < 0 or int(row["unit_price"]) < 0:
                raise PolicyError("invalid_csv")
    except (TypeError, ValueError):
        raise PolicyError("invalid_csv") from None
    return rows


invalid_csv = "date,product,quantity,unit_price\n2026-01-01,A,-2,10000\n"
try:
    validate_input(invalid_csv)
except PolicyError:
    print("Confirmed that negative quantities are rejected")
else:
    raise RuntimeError("Invalid input was not rejected")
validated_rows = validate_input(csv_text)
print("Actual CSV accepted; rows:", len(validated_rows))
if APPROVAL_MODE not in {"interactive", "fixture"}:
    raise ValueError("APPROVAL_MODE must be interactive or fixture")
if REVIEW_MODE not in {"none", "auto_review"}:
    raise ValueError("REVIEW_MODE must be none or auto_review")
using_fixture = APPROVAL_MODE == "fixture"
approver = validation_approval if using_fixture else interactive_approval
print({"approval_mode": APPROVAL_MODE, "review_mode": REVIEW_MODE})
print("Fixture mode: no human input; test approval only." if using_fixture else
      "Interactive mode: user input is required at the training registration step.")


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)

task = (
    "Use Python's standard csv module to read /workspace/sales.csv. "
    "Write /workspace/outputs/summary.json in the requested base format and "
    "/workspace/outputs/report.md in Korean starting with '# 매출 분석 보고서'. "
    "Include row count, total quantity, total revenue as digits, and each product's revenue. "
    "Read both files back. " + METRICS_CONTRACT
)
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [{"type": "inline", "path": "/workspace/sales.csv",
               "data": base64.b64encode(csv_bytes).decode("ascii")}],
}


### 사용자의 판단이 API 도구 결과로 이어지는 흐름

```text
publish_report request → requires_action → arguments/role/channel/file checks
  external → success=False without an approval prompt
  training → input() → Approval → register or reject → success=True/False
→ return tool_result with original turn_id/call_id → model completes the Turn
```

스키마는 첫 Turn부터 제공하지만 보고서 다운로드·제목/바이트 확인 뒤에만 핸들러를 연결합니다. 첫 Turn에서 뜻밖의 호출이 오면 등록하지 않고 오류 결과를 보냅니다. callback은 API 서버가 아닌 노트북 Python에서 실행하며 입력 자체가 API 요청은 아닙니다.

같은 session/turn/call의 결과는 캐시에서 읽어 같은 ID로 반환합니다. 새 Call까지 사용자 결정을 재사용하거나 분산 exactly-once를 보장하지 않습니다. 실행 후 캐시 저장 전에 중단되면 부작용 상태를 먼저 조사합니다.


In [ ]:
def respond_to_pending(client, event, handlers, cache):
    for action in event.session.required_actions:
        if action.type != "function_call":
            raise RuntimeError("Unsupported required action: " + action.type)
        identity = [event.session.id, action.turn_id, action.call_id]
        request = {"name": action.name, "arguments": action.arguments}
        response = cache.load(identity, request)
        print({"pending_call": identity, "tool": action.name, "cache_hit": response is not None}, flush=True)
        if response is None:
            response = {
                "type": "agent.session.input.tool_result",
                "turn_id": action.turn_id,
                "call_id": action.call_id,
            }
            try:
                if action.name not in handlers or not isinstance(action.arguments, dict):
                    raise PolicyError("unknown_function_or_invalid_arguments")
                output = handlers[action.name](action.arguments)
                response.update(success=True, output=json.dumps(output, ensure_ascii=False))
            except PolicyError as tool_error:
                response.update(success=False, error=str(tool_error))
            except Exception:
                response.update(success=False, error="tool_execution_failed")
            cache.save(identity, request, response)
        client.beta.agents.sessions.events.create(event.session.id, events=[response], timeout=30)


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


## 3. 6단계 실행 상태 준비

한 실행에서 한 번만 상태를 준비합니다. 중간에 다시 초기화하면 자원 식별자를 잃습니다. 결과와 복구 기록은 outputs/extensions/<chapter>/<run_id>/에, 다운로드는 그 아래 downloads/<turn_id>/에 보관합니다. 입력 전송 시점에도 실제 CSV 검사를 다시 적용합니다.

시계는 각 API 단계 제출 직전에 시작하며 입력 대기 시간은 제외합니다. 이벤트 도착 시 확인하는 600초 제한이지 무응답 중 강제 중단하는 전체 제한은 아닙니다. 중간에 멈추면 마지막 정리 셀을 실행하고 오류 후에는 새 client와 상태로 시작합니다.


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/extensions" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = ({}, {}, [])
cleanup_errors = []
cache = ResultCache(output_dir / "tool-results")
handlers = {}
analysis = gate = None
decision_label, registered = "not_completed", False
downloads = None
downloads_complete = False
denied = approval_result = None


### 시나리오별 사용자 행동과 예상 결과

| 실행 단계 | 사용자가 할 일 | 완료 확인 |
| --- | --- | --- |
| 1. 보고서 생성 | 입력 없이 대기 | 분석 Turn 완료 |
| 2. 다운로드·검토 | 표시한 보고서와 digest 읽기 | 두 파일·보고서 ID 확인 |
| 3. external 요청 | 입력 없이 대기 | channel_not_allowed·미등록 |
| 4. training 요청 | APPROVE 또는 REJECT 입력 | callback 결정과 도구 결과 |
| 5. 결과 읽기 | 결정·파일·승인 영수증 확인 | 승인 등록 또는 정상 거절 |
| 6. 자원 정리 | 마지막 셀 실행 | 삭제/부재 확인·로컬 결과 보관 |

승인·거절 중 하나를 선택하는 실습이며 두 경로 모두 정상입니다. 다른 경로를 실습하려면 새 client와 상태로 준비부터 시작합니다. 캐시된 같은 Call을 다시 보내 결정을 바꾸지 않습니다. 정상 경로에서 세션은 단계 사이에 유지되며 중간에 멈추면 마지막 정리 셀을 실행합니다. 1–5단계는 오류 정리 context manager, 6단계는 최종 try/finally를 사용합니다. 강제 커널 종료는 자동 정리를 보장하지 않습니다.

external은 허용하지 않은 예시 채널이며 실제 외부 서비스에 연결되어 있지 않습니다. 보고서를 작성할 수 있다는 것과 모든 채널에 등록할 수 있다는 것은 다른 권한입니다. API 요청·이벤트·다운로드·도구 결과 제출은 아래 코드에서 직접 확인합니다.


### 1. 보고서 생성

사용자 입력 없이 API의 분석 결과를 기다립니다. 이 셀은 세션을 생성하므로 준비 셀부터 순서대로 실행하고, 이미 완료한 생성 셀만 반복 실행하지 마세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    validate_input(csv_bytes.decode("utf-8"))
    started = time.monotonic()
    handlers = {}

    # 1. Analyze the CSV and generate the report
    print("[1/6] Generating the report...", flush=True)
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS, "tools": [PUBLISH_TOOL]},
        environment=environment,
        input=task + " Do not call publish_report in this first task.",
        review=REVIEW_MODE,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                respond_to_pending(client, event, handlers, cache)
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        analysis = stream.get_final_result()
        sessions.setdefault(analysis.session_id, {"deleted": False})

        # Retrieve the completed turn
        turn = client.beta.agents.sessions.turns.retrieve(
            analysis.turn_id, session_id=analysis.session_id, timeout=15
        )
        if analysis.turn.status != "completed" or turn.status != "completed":
            raise RuntimeError("Root turn completion was not confirmed")

    print({"analysis_turn": analysis.turn_id, "status": turn.status, "review_mode": REVIEW_MODE})
    print(json.dumps(json.loads(analysis.output_text), indent=2))


### 2. 보고서 다운로드·검토와 정책 연결

완료한 분석 Turn의 두 파일을 내려받고 artifact ID·Turn·원격/로컬 경로를 확인합니다. 모두 받은 뒤 summary와 보고서를 이 노트북에서 읽습니다. 표시한 보고서의 바이트와 digest를 관문에 고정한 후 핸들러를 연결합니다. 아직 사용자 입력이나 등록은 하지 않습니다.

제목 형식·바이트 확인은 내용의 정답/보안 검증을 대신하지 않습니다. 계산값·보고서 대조는 검증 실행 사본에서만 수행합니다. 파일을 검토한 뒤 다음 단계로 진행하세요.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if analysis is None or analysis.turn.status != "completed":
        raise RuntimeError("Complete report generation before downloading.")
    print("[2/6] Downloading and reviewing report files...", flush=True)
    downloads_complete = False
    downloads = output_dir / "downloads" / analysis.turn_id
    downloads.mkdir(parents=True, exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(analysis)
    for name in ("summary.json", "report.md"):
        local_path = downloads / name
        artifact = scoped.download(f"/workspace/outputs/{name}", to=local_path, timeout=30)
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path)})
    downloads_complete = True
    gate = PublicationGate("sales-2026-01", downloads / "report.md", output_dir / "publications",
                           approver=approver, actor_role="analyst")
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    print("Downloaded summary.json:", json.dumps(downloaded_summary, indent=2))
    display(Markdown(gate.reviewed_bytes.decode("utf-8")))
    print({"report_id": gate.report_id, "sha256": gate.digest, "channel": "training"})
    print("Review the report. The next external-channel probe needs no input.", flush=True)
    handlers = {"publish_report": gate.publish}


### 3. 허용하지 않은 등록 채널 차단하기

앞 단계에서 보고서를 다운로드하고 `PublicationGate`를 연결했습니다. 이제 모델이 보고서 등록을 요청하더라도 애플리케이션의 채널 정책을 따라야 하는지 확인합니다.

이 단계에서는 테스트를 위해 모델에 `publish_report`를 `channel="external"`로 한 번 호출하도록 요청합니다. `external`은 허용하지 않은 채널을 나타내는 예시 값이며, 실제 외부 서비스나 전송 도구에 연결되어 있지 않습니다. 앞 단계의 보고서를 수정하지 않고 금지된 요청이 차단되는지만 확인합니다.

요청은 다음 순서로 처리됩니다.

1. 모델이 `publish_report` 호출을 요청합니다.
2. Agents API가 `agent.session.requires_action` 이벤트로 필요한 도구 호출을 전달합니다.
3. 애플리케이션의 `respond_to_pending()`이 요청을 `PublicationGate.publish()`로 전달합니다.
4. 역할·보고서 ID 검사를 통과한 뒤 채널 검사에서 `channel_not_allowed`를 발생시킵니다.
5. 애플리케이션이 `success=False`인 도구 결과를 API에 반환하고, 모델이 거절 결과를 응답합니다.

실제 차단 지점은 앞서 정의한 다음 조건입니다.

```python
if arguments["channel"] != "training":
    raise PolicyError("channel_not_allowed")
```

채널 검사는 승인 함수보다 먼저 실행됩니다. 따라서 이 단계에서는 **사용자 입력을 요청하지 않으며 보고서도 등록하지 않습니다.** 사람에게 승인 여부를 묻기 전에 애플리케이션이 허용하는 작업인지부터 검사하는 구조입니다. 사용자가 승인하더라도 이 채널 제한을 우회하도록 구현되어 있지 않습니다.

**사용자가 확인할 결과:** 입력하지 않고 기다린 뒤 `channel_not_allowed`가 반환되고 등록 파일이 생성되지 않았는지 확인하세요. 코드가 거절 사유·등록 영수증 부재·등록 디렉터리 부재를 확인합니다. 이 거절은 실습 실패가 아니라 의도한 정책 차단 결과입니다. 예상 거절 확인 출력이 나오면 다음 셀로 진행하세요.

다음 단계에서는 같은 보고서를 허용된 `training` 채널로 요청합니다. 채널 검사를 통과한 뒤 사용자 승인 함수가 호출되는 차이를 확인합니다.

| 단계 | 채널 | 판단 주체 | 사용자 입력 | 예상 결과 |
| --- | --- | --- | --- | --- |
| 3번 | `external` | 애플리케이션의 채널 정책 | 없음 | 승인 함수를 호출하기 전에 거절 |
| 4번 | `training` | 정책 검사 후 사용자 승인 함수 | `APPROVE` 또는 `REJECT` | 승인 시 등록, 거절 시 미등록 |


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete or gate is None:
        raise RuntimeError("Download and review the report before requesting registration.")
    started = time.monotonic()
    # Apply channel policy before any approval prompt. Reject publishing to a disallowed channel
    print("[3/6] Testing the blocked external channel; no approval input is needed.", flush=True)
    with client.beta.agents.sessions.stream(
        analysis.session_id,
        input=(
            "This is a training policy probe. Call "
            "publish_report exactly once with report_id "
            "sales-2026-01 and channel external. Do not "
            "change channel or retry if denied. On error, "
            "return only JSON with "
            "publication_status=denied and reason=the "
            "error code."
        ),
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                respond_to_pending(client, event, handlers, cache)
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        denied = stream.get_final_result()
        sessions.setdefault(denied.session_id, {"deleted": False})

        # Retrieve the completed turn
        turn = client.beta.agents.sessions.turns.retrieve(
            denied.turn_id, session_id=denied.session_id, timeout=15
        )
        if denied.turn.status != "completed" or turn.status != "completed":
            raise RuntimeError("Root turn completion was not confirmed")
    denial = json.loads(denied.output_text)
    assert denial["publication_status"] == "denied" and denial["reason"] == "channel_not_allowed"
    assert not gate.receipts and (not gate.destination.exists())

    print("\n[3/6] Expected denial confirmed: channel_not_allowed", flush=True)


### 4. 사용자 승인 또는 거절

보고서·ID·digest를 검토한 뒤 정확한 APPROVE 또는 REJECT를 입력하고 Enter를 누르세요. VS Code에서는 보통 창 상단 입력 상자, Jupyter/Colab에서는 해당 입력 UI를 사용합니다. 셀 아래에 창이 없어도 상단을 확인하세요. 정확한 APPROVE만 승인하며 다른 입력은 거절합니다. 중단 버튼이나 Escape는 거절 입력을 대신하지 않습니다.

프롬프트가 나타난 뒤 busy 상태는 사용자 입력 대기입니다. input에는 자동 timeout이 없고 로컬 도구 처리·승인 대기 시간은 이벤트 기반 시간 예산에서 제외합니다. 프롬프트가 아직 없다면 모델/도구 요청을 기다리는 단계일 수 있습니다. fixture 모드는 입력 없이 테스트 승인하며 사람의 결정을 의미하지 않습니다.

이 셀은 승인 callback과 도구 결과 전송까지 진행합니다. 다음 셀에서 결과 파일과 승인 영수증을 읽고 마지막 정리 셀까지 실행하세요. 두 선택 모두 정상입니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if gate is None or denied is None:
        raise RuntimeError("Complete the policy-denial scenario before requesting a user decision.")
    started = time.monotonic()
    # 5. Request a user decision for the local training channel
    print("[4/6] Requesting training-channel registration...", flush=True)
    print("Report to review:", gate.path, flush=True)
    print("Report ID:", gate.report_id, "SHA256:", gate.digest, flush=True)
    if using_fixture:
        print("Fixture mode: the next call uses test approval without human input.", flush=True)
    else:
        print("The approval prompt appears when publish_report is requested.", flush=True)
    with client.beta.agents.sessions.stream(
        analysis.session_id,
        input=(
            "Call publish_report exactly once with "
            "report_id sales-2026-01 and channel training."
            " The application handles approval; do not "
            "supply any approval flag. On success, return "
            "the exact receipt JSON. On approval_denied, "
            'return exactly {"publication_status":"denied",'
            '"reason":"approval_denied"}. Do not retry or '
            "switch channels after a denial."
        ),
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                approval_started = time.monotonic()
                respond_to_pending(client, event, handlers, cache)
                started += time.monotonic() - approval_started
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        approval_result = stream.get_final_result()
        sessions.setdefault(approval_result.session_id, {"deleted": False})

        # Retrieve the completed turn
        turn = client.beta.agents.sessions.turns.retrieve(
            approval_result.turn_id, session_id=approval_result.session_id, timeout=15
        )
        if approval_result.turn.status != "completed" or turn.status != "completed":
            raise RuntimeError("Root turn completion was not confirmed")

    if gate.decision is None:
        raise RuntimeError("The approval callback was not called")
    decision_label = "approved" if gate.decision.approved else "rejected"
    registered = bool(gate.receipts)
    print("User callback decision:", decision_label, flush=True)


### 5. 사용자 결정과 로컬 등록 결과 읽기

이 셀은 새 모델 요청이나 승인 입력 없이 로컬 결과를 읽습니다. 승인 시에는 등록 보고서와 승인 영수증을 확인합니다. 거절 시에는 approval_denied와 등록 파일 부재를 확인합니다. 모델의 완료 문구 대신 callback 결정·도구 결과·실제 파일을 함께 읽습니다.

| 기록 | 언제 생성되나요? |
| --- | --- |
| .approval.json 승인 영수증 | 사용자 또는 fixture 승인 후 로컬 등록 시, 수동 실행에도 생성 |
| tool-results 캐시 | 처리한 동일 Call의 도구 결과를 보관 |
| summary.json 자동 검증 기록 | 별도 검증 도구의 실행 사본에서만 생성 |

다운로드한 summary.json은 분석 결과이며 자동 검증 기록과 다른 경로의 파일입니다. 승인 영수증의 fixture origin은 실제 사람이 승인했다는 뜻이 아닙니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if approval_result is None or gate is None or gate.decision is None:
        raise RuntimeError("Complete the approval request before reading its result.")
    outcome = json.loads(approval_result.output_text)
    if gate.decision is None:
        raise RuntimeError("The approval callback was not called")
    if gate.decision.approved:
        assert outcome["publication_status"] == "published" and outcome["sha256"] == gate.digest
        assert gate.receipts and (
            gate.destination / f"{gate.digest}.md"
        ).read_bytes() == (downloads / "report.md").read_bytes()
        assert outcome["approval_origin"] == ("fixture" if using_fixture else "interactive")
        decision_label, registered = "approved", True
    else:
        assert outcome == {"publication_status": "denied", "reason": "approval_denied"}
        assert not gate.receipts and not gate.destination.exists()
        decision_label, registered = "rejected", False
    print("\nApproval decision:", decision_label, flush=True)
    print("Report registered:", "yes" if registered else "no", flush=True)

    if registered:
        receipt_path = gate.destination / f"{gate.digest}.approval.json"
        approval_receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        print("Approval receipt:", json.dumps(approval_receipt, indent=2))
    print("[5/6] Decision and local registration inspected. Run cleanup next.", flush=True)


### 6. 자원 정리·최종 요약

마지막에 반드시 실행합니다. 중간에 멈출 때도 이 셀을 실행하세요. 사용자 결정 전이면 not_completed이며, 승인 요청이 완료됐다면 결과 읽기 셀을 건너뛰어도 실제 결정·등록 상태를 요약합니다. 정상 정리 후 client는 닫히므로 같은 client로 API 확인을 다시 시도하지 않습니다. 오류 시 앞 단계에서 정리를 시도한 상태를 확인하고 미정리 자원은 복구 기록과 새 client로 조사합니다. API상 삭제/404 확인은 물리적 sandbox 정리의 동기 완료를 독립 증명하지 않습니다.


In [ ]:
try:
    if analysis is not None and not sessions[analysis.session_id].get("verified_absent"):
        effective = client.beta.agents.sessions.retrieve(analysis.session_id, timeout=15)
        if effective.review != REVIEW_MODE:
            raise RuntimeError("Effective review mode differs from the requested mode")
    print("[6/6] Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})
print({"approval_decision": decision_label, "report_registered": registered,
       "session_cleanup": "verified" if sessions else "not_needed"})
if registered:
    print("Registered report:", gate.destination / f"{gate.digest}.md")
    print("Approval receipt:", gate.destination / f"{gate.digest}.approval.json")
elif decision_label == "rejected":
    print("Expected user rejection confirmed; no registration files were written.")
else:
    print("The approval scenario was not completed; prepare a new run to try again.")


## 완료 기준과 승인 검증의 의미

거절 요청은 실제 required action과 success=False 응답으로 확인하며 파일이 생기지 않아야 합니다.
허용 요청은 승인 digest와 저장 파일 바이트를 대조합니다. 모델의 '게시 완료' 문구만 확인하지 않습니다.
자동 검증의 fixture 승인은 테스트 입력이며 실제 사람이 승인했다는 감사 기록이 아닙니다.
노트북 직접 실행은 기본 interactive 승인 입력을 사용합니다. 사용자 거절도 정상 완료이며 `approval_denied`와 등록 파일 부재를 확인합니다. 실제 정책 오류·파일 불일치·API 실패와 구분합니다. 6번 정리 셀에서 승인 결정·등록 여부·자원 정리 결과를 함께 확인하세요.

이 코드는 인증 시스템이나 전체 DLP를 구현한 것이 아닙니다. 역할은 신뢰하는 애플리케이션 문맥에서 주입합니다.
프로덕션에서는 인증된 사용자, 정책 저장소, 승인 UI와 부작용의 트랜잭션·복구를 별도로 연결합니다.

[15장 Multi-agent](15_multi_agent.ipynb)와 조합한다면 메인이 통합한 최종 보고서에 이 장의 검증·승인을 적용하는 순서를 권장합니다. 공식 문서상 하위 에이전트는 Function Tool을 지원하지 않으므로 승인 핸들러를 하위 담당에 그대로 연결하지 않습니다. 현재 15장은 보고서 생성·다운로드까지만 수행하며 승인과의 조합은 실제 실행 검증 범위에 포함되지 않습니다.

### 다음 학습 경로

기능 확장 과정을 마쳤습니다. 사내 환경을 구성하려면 [12장 Self-hosted](12_self_hosted.ipynb)→13장 관측→14장 복구로 진행하세요. 역할 분담을 먼저 경험하려면 [15장 Multi-agent](15_multi_agent.ipynb)로 이동할 수 있습니다. 15장은 3·5장 지식으로 실행할 수 있으며 승인 조합은 선택 확장입니다.


## 자원 정리와 실행 기록

이번 실행의 소유 세션만 취소·삭제·재조회 404로 확인하며 필요한 파일은 먼저 다운로드합니다. 등록된 로컬 보고서·승인 영수증·캐시는 세션 삭제 후에도 남습니다. 별도 영구 Agent나 외부 게시 자원은 만들지 않습니다.

실행 사본은 outputs/extensions-validation/, 결과/복구 기록은 outputs/extensions/에 있습니다. 일반 Jupyter 실행에도 승인 시 승인 영수증을 쓰지만 자동 검증 기록은 검증 도구만 생성합니다. 강제 종료에서는 finally가 실행되지 않을 수 있으므로 creation.json metadata와 Session ID로 소유 자원을 복구합니다.

같은 session/turn/call 캐시는 이미 처리한 결과를 재전송하는 용도입니다. 저장 전 중단된 부작용은 조사 후 재개하며 새 Call에 대해 승인 결정을 바꾸거나 재사용하는 것은 별도 정책입니다. [14장 관찰자 재연결](14_operations.ipynb)은 원래 Turn을 관찰하는 복구이며 승인 대기를 자동 재개하는 예제가 아닙니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Functions](https://developers.openai.com/api/docs/guides/agents-api/tools/functions) — 함수 정의·required action·도구 결과 반환과 복구.
- [Automatic review](https://developers.openai.com/api/docs/guides/agents-api/sessions/approvals) — 지원되는 실행 권한 요청에 대한 review 설정.
- [Sandbox security](https://developers.openai.com/api/docs/guides/agents-api/environments/security) — 격리·네트워크·실행 권한 검토·인증 정보 분리.

이 장의 입력·출력 검증과 업무용 사람 승인은 애플리케이션에서 구현합니다. Automatic review는 지원되는 실행 권한 요청의 검토이며 보고서 게시 승인을 대신하지 않습니다.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
